# Notebook Overview — Generate Reports

## Purpose

This notebook evaluates VideoQA experiment results and generates performance reports, analysis summaries, and visualizations for baseline, Retrieval-Augmented Generation (RAG), and iterative RAG workflows. The notebook loads prediction results, experiment summaries, runtime statistics, evidence metadata, and evaluation reference data produced by previous notebooks and generates metrics, analysis tables, visualizations, and reporting artifacts for experiment assessment and comparison.

## Inputs

* VideoQA prediction results
  * outputs/baseline/baseline_predictions.csv
* Experiment summary results
  * outputs/baseline/baseline_summary.csv
* Evidence summary results
  * outputs/evidence/reports/evidence_summary.csv
* NExT-QA question annotations
* Evaluation reference datasets
* Project configuration settings

## Outputs

* Evaluation metrics tables
* Prediction analysis summaries
* Runtime analysis summaries
* Evidence utilization summaries
* Performance visualizations
* Experiment evaluation reports
* Saved reporting artifacts
* Displayed charts, tables, and statistics

## Workflow

The workflow begins by configuring the notebook environment and loading experiment results together with evaluation reference data. Input files are validated before evaluation metrics are compiled. The notebook then generates prediction analysis, runtime analysis, and evidence utilization summaries. Visualizations are created to support experiment interpretation and comparison. Finally, evaluation results, analysis summaries, and reporting artifacts are saved and displayed for review and documentation.


### 🔷 Step 1 — Clone Required Repository Files

* Clone the project repository using sparse checkout to minimize download size and runtime initialization overhead.
* Authenticate access to the private GitHub repository using a fine-grained access token stored in Google Colab Secrets.
* Configure the local notebook workspace and change to the repository working directory.
* Verify that required repository files and directories are available for subsequent notebook execution.
* Optionally display repository paths, directory contents, and cloned files when `VERBOSE=True`.


In [ ]:
# ============================================================
# Step 1: Clone Required Repository Files
# ============================================================

VERBOSE = True

import os
from google.colab import userdata

REPO_NAME = "iterative-video-rag"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

# ------------------------------------------------------------
# Retrieve GitHub Token from Colab Secrets
# ------------------------------------------------------------

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets."
    )

repo_url = (
    f"https://{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

# ------------------------------------------------------------
# Move to Base Directory
# ------------------------------------------------------------
%cd {REPO_BASE_DIR}

# ------------------------------------------------------------
# Clone Repository if Needed
# ------------------------------------------------------------

if not os.path.exists(REPO_DIR):
    if VERBOSE:
        print("Cloning required repository directories...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}
    %cd {REPO_DIR}
    !git sparse-checkout init --cone
    !git sparse-checkout set \
        src \
        datasets \
        outputs
    !git checkout --quiet main

else:
    if VERBOSE:
        print(f"Repository already exists: {REPO_DIR}")
    %cd {REPO_DIR}

# ------------------------------------------------------------
# Verify Repository Structure
# ------------------------------------------------------------
required_paths = [
    "src",
    "datasets",
    "outputs",
    "datasets/NExT-QA",
    "datasets/NExT-QA/questions",
]

for path in required_paths:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required path not found: {path}"
        )

# ------------------------------------------------------------
# Verify Notebook 02 Outputs
# ------------------------------------------------------------
required_files = [
    "outputs/baseline/reports/baseline_predictions.csv",
    "outputs/baseline/reports/baseline_summary.csv",
    "outputs/evidence/metadata/evidence_metadata.csv",
    "outputs/evidence/reports/evidence_summary.csv",
    "datasets/NExT-QA/questions/val.csv",
]

for file_path in required_files:

    if not os.path.exists(file_path):

        raise FileNotFoundError(
            f"Required file not found: {file_path}"
        )

print("Repository setup complete.")

# ------------------------------------------------------------
# Display Repository Summary
# ------------------------------------------------------------

if VERBOSE:
    print(f"\nCurrent directory: {os.getcwd()}")
    print("\nRepository directories:")
    !find src datasets outputs \
        -maxdepth 2 \
        -type d \
        ! -path "*/__pycache__*" | sort

    print("\nVerified Notebook 02 Outputs")
    print("-" * 60)

    for file_path in required_files:
        file_size_mb = (
            os.path.getsize(file_path)
            / (1024 * 1024)
        )
        print(
            f"{file_path:<60} "
            f"{file_size_mb:8.2f} MB"
        )



### 🔷 Step 2 — Load Evaluation Data

* Load baseline VideoQA prediction results generated by Notebook 03.
* Load experiment summary statistics and runtime metrics.
* Load evidence metadata summary information generated by Notebook 02.
* Load NExT-QA annotation records required for evaluation and reporting.
* Display dataset sizes and basic file statistics to verify successful loading.


In [ ]:
# ============================================================
# Step 2: Load Evaluation Data
# ============================================================

import pandas as pd

print("Loading evaluation data...\n")

# ------------------------------------------------------------
# Load Baseline Experiment Results
# ------------------------------------------------------------
baseline_predictions_df = pd.read_csv(
    "outputs/baseline/reports/baseline_predictions.csv"
)
baseline_summary_df = pd.read_csv(
    "outputs/baseline/reports/baseline_summary.csv"
)

# ------------------------------------------------------------
# Load Evaluation Reference Data
# ------------------------------------------------------------
val_annotations_df = pd.read_csv(
    "datasets/NExT-QA/questions/val.csv"
)
evidence_metadata_df = pd.read_csv(
    "outputs/evidence/metadata/evidence_metadata.csv"
)
evidence_summary_df = pd.read_csv(
    "outputs/evidence/reports/evidence_summary.csv"
)

# ------------------------------------------------------------
# Display Dataset Information
# ------------------------------------------------------------
print("Loaded Evaluation Data")
print("-" * 60)
print(
    f"Baseline Predictions      : "
    f"{len(baseline_predictions_df):,} records"
)
print(
    f"Baseline Summary          : "
    f"{len(baseline_summary_df):,} records"
)
print(
    f"Validation Annotations    : "
    f"{len(val_annotations_df):,} records"
)
print(
    f"Evidence Metadata         : "
    f"{len(evidence_metadata_df):,} records"
)
print(
    f"Evidence Summary          : "
    f"{len(evidence_summary_df):,} records"
)

# ------------------------------------------------------------
# Display Available Columns
# ------------------------------------------------------------
print("\nBaseline Prediction Columns")
print("-" * 60)
print(list(baseline_predictions_df.columns))

print("\nBaseline Summary Columns")
print("-" * 60)
print(list(baseline_summary_df.columns))

# ------------------------------------------------------------
# Preview Loaded Data
# ------------------------------------------------------------
print("\nBaseline Predictions Preview")
display(baseline_predictions_df.head())

print("\nBaseline Summary Preview")
display(baseline_summary_df.head())



### 🔷 Step 3 — Validate Evaluation Inputs

* Verify that all required evaluation input files are present.
* Confirm required columns exist in prediction and summary datasets.
* Validate record counts and data integrity.
* Check for missing values, duplicate records, and invalid entries.
* Report validation results before evaluation processing begins.


In [ ]:
# ============================================================
# Step 3: Validate Evaluation Inputs
# ============================================================

print("Validating evaluation inputs...\n")

validation_passed = True

# ------------------------------------------------------------
# Verify Required DataFrames
# ------------------------------------------------------------
required_dataframes = {
    "Baseline Predictions": baseline_predictions_df,
    "Baseline Summary": baseline_summary_df,
    "Validation Annotations": val_annotations_df,
    "Evidence Metadata": evidence_metadata_df,
    "Evidence Summary": evidence_summary_df,
}

print("Dataset Validation")
print("-" * 60)

for name, df in required_dataframes.items():
    record_count = len(df)
    print(
        f"{name:<25}: "
        f"{record_count:>8,} records"
    )
    if record_count == 0:
        validation_passed = False
        print(f"  ERROR: {name} contains no records")

# ------------------------------------------------------------
# Verify Required Prediction Columns
# ------------------------------------------------------------
required_prediction_columns = [
    "video",
    "question",
    "ground_truth",
    "prediction",
    "evidence_record_count",
]

missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in baseline_predictions_df.columns
]

# ------------------------------------------------------------
# Verify Required Summary Columns
# ------------------------------------------------------------
required_summary_columns = [
    "metric",
    "value",
]

missing_summary_columns = [
    column
    for column in required_summary_columns
    if column not in baseline_summary_df.columns
]

# ------------------------------------------------------------
# Report Missing Columns
# ------------------------------------------------------------
print("\nColumn Validation")
print("-" * 60)

if missing_prediction_columns:
    validation_passed = False
    print(
        "Missing Prediction Columns:"
    )
    for column in missing_prediction_columns:
        print(f"  {column}")
else:
    print(
        "Prediction columns validated."
    )

if missing_summary_columns:
    validation_passed = False
    print(
        "Missing Summary Columns:"
    )
    for column in missing_summary_columns:
        print(f"  {column}")
else:
    print(
        "Summary columns validated."
    )

# ------------------------------------------------------------
# Verify Summary Metrics
# ------------------------------------------------------------
required_metrics = [
    "total_predictions",
    "valid_predictions",
    "missing_predictions",
    "empty_predictions",
    "error_predictions",
    "unique_videos",
    "elapsed_time_seconds",
]

available_metrics = set(
    baseline_summary_df["metric"]
)

missing_metrics = [
    metric
    for metric in required_metrics
    if metric not in available_metrics
]

print("\nMetric Validation")
print("-" * 60)

if missing_metrics:
    validation_passed = False
    print(
        "Missing Summary Metrics:"
    )
    for metric in missing_metrics:
        print(f"  {metric}")
else:
    print(
        "Summary metrics validated."
    )

# ------------------------------------------------------------
# Final Validation Status
# ------------------------------------------------------------

print("\nValidation Results")
print("-" * 60)

print(
    f"Validation Passed : "
    f"{validation_passed}"
)

if not validation_passed:
    raise ValueError(
        "Evaluation input validation failed."
    )



### 🔷 Step 4 — Compile Evaluation Metrics

* Calculate overall experiment performance statistics.
* Compile prediction counts, video counts, and evaluation sample totals.
* Summarize baseline inference results and runtime measurements.
* Generate evaluation metric tables for reporting and comparison.
* Prepare evaluation metrics for visualization and export.


In [ ]:
# ============================================================
# Step 4: Compile Evaluation Metrics
# ============================================================

print("Compiling evaluation metrics...\n")

# ------------------------------------------------------------
# Convert Summary Tables to Dictionaries
# ------------------------------------------------------------

baseline_metrics = dict(
    zip(
        baseline_summary_df["metric"],
        baseline_summary_df["value"]
    )
)

evidence_metrics = dict(
    zip(
        evidence_summary_df["metric"],
        evidence_summary_df["value"]
    )
)

# ------------------------------------------------------------
# Build Consolidated Evaluation Metrics
# ------------------------------------------------------------

evaluation_metrics = [
    {
        "metric": "total_predictions",
        "value": baseline_metrics.get(
            "total_predictions"
        )
    },
    {
        "metric": "valid_predictions",
        "value": baseline_metrics.get(
            "valid_predictions"
        )
    },
    {
        "metric": "missing_predictions",
        "value": baseline_metrics.get(
            "missing_predictions"
        )
    },
    {
        "metric": "empty_predictions",
        "value": baseline_metrics.get(
            "empty_predictions"
        )
    },
    {
        "metric": "error_predictions",
        "value": baseline_metrics.get(
            "error_predictions"
        )
    },
    {
        "metric": "unique_videos_evaluated",
        "value": baseline_metrics.get(
            "unique_videos"
        )
    },
    {
        "metric": "average_evidence_records_per_sample",
        "value": baseline_metrics.get(
            "average_evidence_records_per_sample"
        )
    },
    {
        "metric": "elapsed_time_seconds",
        "value": baseline_metrics.get(
            "elapsed_time_seconds"
        )
    },
    {
        "metric": "average_time_per_sample_seconds",
        "value": baseline_metrics.get(
            "average_time_per_sample_seconds"
        )
    },
    {
        "metric": "projected_validation_runtime_minutes",
        "value": baseline_metrics.get(
            "projected_validation_runtime_minutes"
        )
    },
    {
        "metric": "projected_full_dataset_runtime_hours",
        "value": baseline_metrics.get(
            "projected_full_dataset_runtime_hours"
        )
    },
    {
        "metric": "evidence_record_count",
        "value": evidence_metrics.get(
            "evidence_record_count"
        )
    },
    {
        "metric": "unique_video_count",
        "value": evidence_metrics.get(
            "unique_video_count"
        )
    },
    {
        "metric": "average_evidence_per_video",
        "value": evidence_metrics.get(
            "average_evidence_per_video"
        )
    },
    {
        "metric": "segment_strategy",
        "value": evidence_metrics.get(
            "segment_strategy"
        )
    },
    {
        "metric": "default_segment_duration_sec",
        "value": evidence_metrics.get(
            "default_segment_duration_sec"
        )
    },
    {
        "metric": "validation_passed",
        "value": evidence_metrics.get(
            "validation_passed"
        )
    },
]

evaluation_metrics_df = pd.DataFrame(
    evaluation_metrics
)

# ------------------------------------------------------------
# Display Evaluation Metrics
# ------------------------------------------------------------

print("Compiled Evaluation Metrics")
print("-" * 60)

display(evaluation_metrics_df)

print(
    f"\nTotal Metrics Compiled : "
    f"{len(evaluation_metrics_df):,}"
)



### 🔷 Step 5 — Generate Prediction Analysis

* Analyze generated VideoQA predictions.
* Compare predicted answers with ground-truth responses.
* Calculate prediction distribution statistics.
* Summarize answer characteristics and response patterns.
* Generate prediction analysis tables for review.


In [ ]:
# ============================================================
# Step 5: Generate Prediction Analysis
# ============================================================

print("Generating prediction analysis...\n")

# ------------------------------------------------------------
# Create Prediction Analysis Summary
# ------------------------------------------------------------

prediction_analysis = {
    "total_prediction_records":
        len(baseline_predictions_df),

    "unique_videos":
        baseline_predictions_df["video"].nunique(),

    "questions_with_predictions":
        baseline_predictions_df["prediction"].notna().sum(),

    "missing_predictions":
        baseline_predictions_df["prediction"].isna().sum(),

    "empty_predictions":
        (
            baseline_predictions_df["prediction"]
            .fillna("")
            .astype(str)
            .str.strip()
            == ""
        ).sum(),

    "average_question_length_words":
        (
            baseline_predictions_df["question"]
            .fillna("")
            .astype(str)
            .str.split()
            .str.len()
            .mean()
        ),

    "average_ground_truth_length_words":
        (
            baseline_predictions_df["ground_truth"]
            .fillna("")
            .astype(str)
            .str.split()
            .str.len()
            .mean()
        ),

    "average_prediction_length_words":
        (
            baseline_predictions_df["prediction"]
            .fillna("")
            .astype(str)
            .str.split()
            .str.len()
            .mean()
        ),
}

prediction_analysis_df = pd.DataFrame(
    list(prediction_analysis.items()),
    columns=["metric", "value"]
)

# ------------------------------------------------------------
# Display Prediction Analysis
# ------------------------------------------------------------

print("Prediction Analysis Summary")
print("-" * 60)
display(prediction_analysis_df)

# ------------------------------------------------------------
# Display Sample Predictions
# ------------------------------------------------------------

print("\nSample Predictions")
print("-" * 60)

display(
    baseline_predictions_df[
        [
            "video",
            "question",
            "ground_truth",
            "prediction",
            "evidence_record_count",
        ]
    ].head(10)
)



### 🔷 Step 6 — Generate Runtime Analysis

* Analyze experiment execution performance.
* Summarize total runtime and average inference time per sample.
* Estimate processing requirements for larger evaluation runs.
* Calculate projected runtimes for validation and full-dataset execution.
* Generate runtime analysis tables for reporting.


In [ ]:
# ============================================================
# Step 6: Generate Runtime Analysis
# ============================================================

print("Generating runtime analysis...\n")

# ------------------------------------------------------------
# Helper Function for Summary Metric Lookup
# ------------------------------------------------------------

def get_metric(metric_name, default=None):
    metric_rows = baseline_summary_df[
        baseline_summary_df["metric"] == metric_name
    ]

    if metric_rows.empty:
        return default

    return metric_rows["value"].iloc[0]

# ------------------------------------------------------------
# Extract Runtime Metrics
# ------------------------------------------------------------

runtime_analysis = {
    "elapsed_time_seconds":
        get_metric("elapsed_time_seconds"),

    "average_time_per_sample_seconds":
        get_metric("average_time_per_sample_seconds"),

    "projected_validation_runtime_minutes":
        get_metric("projected_validation_runtime_minutes"),

    "projected_full_dataset_runtime_hours":
        get_metric("projected_full_dataset_runtime_hours"),

    "total_predictions":
        get_metric("total_predictions"),

    "valid_predictions":
        get_metric("valid_predictions"),
}

runtime_analysis_df = pd.DataFrame(
    list(runtime_analysis.items()),
    columns=["metric", "value"]
)

# ------------------------------------------------------------
# Add Human-Readable Runtime Values
# ------------------------------------------------------------

elapsed_seconds = float(
    runtime_analysis["elapsed_time_seconds"]
)

average_seconds = float(
    runtime_analysis["average_time_per_sample_seconds"]
)

validation_minutes = float(
    runtime_analysis["projected_validation_runtime_minutes"]
)

full_dataset_hours = float(
    runtime_analysis["projected_full_dataset_runtime_hours"]
)

runtime_summary_df = pd.DataFrame(
    [
        {
            "runtime_metric": "Elapsed runtime",
            "value": elapsed_seconds,
            "unit": "seconds",
        },
        {
            "runtime_metric": "Average runtime per sample",
            "value": average_seconds,
            "unit": "seconds/sample",
        },
        {
            "runtime_metric": "Projected validation split runtime",
            "value": validation_minutes,
            "unit": "minutes",
        },
        {
            "runtime_metric": "Projected full dataset runtime",
            "value": full_dataset_hours,
            "unit": "hours",
        },
    ]
)

# ------------------------------------------------------------
# Display Runtime Analysis
# ------------------------------------------------------------

print("Runtime Analysis Summary")
print("-" * 60)

display(runtime_summary_df)

print("\nRuntime Interpretation")
print("-" * 60)
print(
    f"The baseline run processed "
    f"{int(runtime_analysis['valid_predictions']):,} valid samples "
    f"in {elapsed_seconds:.2f} seconds."
)

print(
    f"Average runtime was "
    f"{average_seconds:.2f} seconds per sample."
)

print(
    f"Projected runtime for the full validation split is "
    f"{validation_minutes:.2f} minutes."
)

print(
    f"Projected runtime for the full dataset is "
    f"{full_dataset_hours:.2f} hours."
)



### 🔷 Step 7 — Generate Evidence Utilization Analysis

* Analyze evidence records used during VideoQA inference.
* Summarize evidence utilization across evaluated samples.
* Compare evidence usage against available repository evidence.
* Calculate evidence coverage and utilization statistics.
* Generate evidence analysis tables for reporting and visualization.


In [ ]:
# ============================================================
# Step 7: Generate Evidence Utilization Analysis
# ============================================================

print("Generating evidence utilization analysis...\n")

# ------------------------------------------------------------
# Analyze Evidence Records Used During Baseline Inference
# ------------------------------------------------------------

evidence_usage_summary = {
    "samples_analyzed":
        len(baseline_predictions_df),

    "total_evidence_records_used":
        baseline_predictions_df["evidence_record_count"].sum(),

    "average_evidence_records_per_sample":
        baseline_predictions_df["evidence_record_count"].mean(),

    "minimum_evidence_records_per_sample":
        baseline_predictions_df["evidence_record_count"].min(),

    "maximum_evidence_records_per_sample":
        baseline_predictions_df["evidence_record_count"].max(),

    "median_evidence_records_per_sample":
        baseline_predictions_df["evidence_record_count"].median(),
}

evidence_usage_df = pd.DataFrame(
    list(evidence_usage_summary.items()),
    columns=["metric", "value"]
)

# ------------------------------------------------------------
# Analyze Available Evidence Metadata
# ------------------------------------------------------------

available_evidence_summary = {
    "total_available_evidence_records":
        len(evidence_metadata_df),

    "unique_videos_with_evidence":
        evidence_metadata_df["video_id"].nunique(),

    "average_available_evidence_per_video":
        (
            evidence_metadata_df
            .groupby("video_id")
            .size()
            .mean()
        ),

    "minimum_available_evidence_per_video":
        (
            evidence_metadata_df
            .groupby("video_id")
            .size()
            .min()
        ),

    "maximum_available_evidence_per_video":
        (
            evidence_metadata_df
            .groupby("video_id")
            .size()
            .max()
        ),
}

available_evidence_df = pd.DataFrame(
    list(available_evidence_summary.items()),
    columns=["metric", "value"]
)

available_evidence_df["value"] = (
    available_evidence_df["value"]
    .apply(
        lambda x:
        round(x, 2)
        if isinstance(x, (int, float))
        else x
    )
)

# ------------------------------------------------------------
# Display Evidence Utilization Analysis
# ------------------------------------------------------------

print("Evidence Records Used During Baseline Inference")
print("-" * 60)
display(evidence_usage_df)

print("\nAvailable Evidence Repository Summary")
print("-" * 60)
display(available_evidence_df)

# ------------------------------------------------------------
# Display Evidence Record Count Distribution
# ------------------------------------------------------------

print("\nEvidence Records Per Evaluated Sample")
print("-" * 60)

display(
    baseline_predictions_df[
        [
            "video",
            "question",
            "evidence_record_count",
        ]
    ].sort_values(
        by="evidence_record_count",
        ascending=False
    ).head(10)
)



### 🔷 Step 8 — Create Visualizations

* Generate charts summarizing experiment performance.
* Visualize prediction, runtime, and evidence utilization statistics.
* Create publication-ready figures suitable for reports and presentations.
* Display generated visualizations within the notebook.
* Prepare visualization files for export.


In [ ]:
# ============================================================
# Step 8: Create Visualizations
# ============================================================

import os
import matplotlib.pyplot as plt

print("Creating visualizations...\n")

# ------------------------------------------------------------
# Create Output Directory
# ------------------------------------------------------------

figure_dir = "outputs/evaluation/figures"

os.makedirs(
    figure_dir,
    exist_ok=True
)

generated_figures = []

# ------------------------------------------------------------
# Visualization 1:
# Evidence Records per Sample
# ------------------------------------------------------------

plt.figure(figsize=(8, 4))

plt.hist(
    baseline_predictions_df[
        "evidence_record_count"
    ],
    bins=10
)

plt.title(
    "Evidence Records per Sample"
)

plt.xlabel(
    "Evidence Record Count"
)

plt.ylabel(
    "Number of Samples"
)

evidence_plot_file = os.path.join(
    figure_dir,
    "evidence_usage_distribution.png"
)

plt.tight_layout()
plt.savefig(
    evidence_plot_file,
    dpi=300
)
plt.close()

generated_figures.append(
    evidence_plot_file
)

# ------------------------------------------------------------
# Visualization 2:
# Question vs Prediction Length
# ------------------------------------------------------------

question_lengths = (
    baseline_predictions_df["question"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

prediction_lengths = (
    baseline_predictions_df["prediction"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

length_metrics = {
    "Question Length":
        question_lengths.mean(),

    "Prediction Length":
        prediction_lengths.mean(),
}

plt.figure(figsize=(6, 4))

plt.bar(
    length_metrics.keys(),
    length_metrics.values()
)

plt.title(
    "Average Text Length"
)

plt.ylabel(
    "Average Words"
)

length_plot_file = os.path.join(
    figure_dir,
    "text_length_comparison.png"
)

plt.tight_layout()
plt.savefig(
    length_plot_file,
    dpi=300
)
plt.close()

generated_figures.append(
    length_plot_file
)

# ------------------------------------------------------------
# Visualization 3:
# Runtime Projections
# ------------------------------------------------------------

runtime_metrics = {
    "Baseline Run":
        float(
            baseline_summary_df[
                baseline_summary_df["metric"]
                ==
                "elapsed_time_seconds"
            ]["value"].iloc[0]
        ) / 60,

    "Validation":
        float(
            baseline_summary_df[
                baseline_summary_df["metric"]
                ==
                "projected_validation_runtime_minutes"
            ]["value"].iloc[0]
        ),

    "Full Dataset":
        float(
            baseline_summary_df[
                baseline_summary_df["metric"]
                ==
                "projected_full_dataset_runtime_hours"
            ]["value"].iloc[0]
        ) * 60,
}

plt.figure(figsize=(7, 4))

plt.bar(
    runtime_metrics.keys(),
    runtime_metrics.values()
)

plt.title(
    "Runtime Comparison"
)

plt.ylabel(
    "Minutes"
)

runtime_plot_file = os.path.join(
    figure_dir,
    "runtime_comparison.png"
)

plt.tight_layout()
plt.savefig(
    runtime_plot_file,
    dpi=300
)
plt.close()

generated_figures.append(
    runtime_plot_file
)

# ------------------------------------------------------------
# Display Generated Figures
# ------------------------------------------------------------

print("Generated Figures")
print("-" * 60)

for file_path in generated_figures:

    file_size_kb = (
        os.path.getsize(file_path)
        / 1024
    )

    print(
        f"{os.path.basename(file_path):<40}"
        f"{file_size_kb:8.1f} KB"
    )



### 🔷 Step 9 — Save Evaluation Results

* Save evaluation metric tables to the project output directory.
* Save prediction analysis summaries.
* Save runtime analysis summaries.
* Save evidence utilization summaries.
* Save generated visualization files for future reporting and comparison.


In [ ]:
# ============================================================
# Step 9: Save Evaluation Results
# ============================================================

import os

print("Saving evaluation results...\n")

# ------------------------------------------------------------
# Create Output Directory
# ------------------------------------------------------------

evaluation_report_dir = (
    "outputs/evaluation/reports"
)

os.makedirs(
    evaluation_report_dir,
    exist_ok=True
)

# ------------------------------------------------------------
# Save Evaluation DataFrames
# ------------------------------------------------------------

evaluation_metrics_file = os.path.join(
    evaluation_report_dir,
    "evaluation_metrics.csv"
)

prediction_analysis_file = os.path.join(
    evaluation_report_dir,
    "prediction_analysis.csv"
)

runtime_analysis_file = os.path.join(
    evaluation_report_dir,
    "runtime_analysis.csv"
)

evidence_analysis_file = os.path.join(
    evaluation_report_dir,
    "evidence_analysis.csv"
)

evaluation_metrics_df.to_csv(
    evaluation_metrics_file,
    index=False
)

prediction_analysis_df.to_csv(
    prediction_analysis_file,
    index=False
)

runtime_analysis_df.to_csv(
    runtime_analysis_file,
    index=False
)

evidence_usage_df.to_csv(
    evidence_analysis_file,
    index=False
)

# ------------------------------------------------------------
# Display Saved Files
# ------------------------------------------------------------

saved_files = [
    evaluation_metrics_file,
    prediction_analysis_file,
    runtime_analysis_file,
    evidence_analysis_file,
]

print("Saved Evaluation Files")
print("-" * 60)

for file_path in saved_files:

    file_size_kb = (
        os.path.getsize(file_path)
        / 1024
    )

    print(
        f"{os.path.basename(file_path):<35}"
        f"{file_size_kb:8.1f} KB"
    )



### 🔷 Step 10 — Display Evaluation Results

* Display evaluation metrics generated during analysis.
* Present prediction, runtime, and evidence utilization summaries.
* Display generated visualizations.
* Review key experiment findings and performance statistics.
* Verify evaluation outputs were successfully generated and saved.

In [ ]:
# ============================================================
# Step 10: Display Evaluation Results
# ============================================================

print("Evaluation Results")
print("=" * 60)

# ------------------------------------------------------------
# Display Consolidated Metrics
# ------------------------------------------------------------

print("\nCompiled Evaluation Metrics")
print("-" * 60)

display(evaluation_metrics_df)

# ------------------------------------------------------------
# Display Prediction Analysis
# ------------------------------------------------------------

print("\nPrediction Analysis")
print("-" * 60)

display(prediction_analysis_df)

# ------------------------------------------------------------
# Display Runtime Analysis
# ------------------------------------------------------------

print("\nRuntime Analysis")
print("-" * 60)

display(runtime_summary_df)

# ------------------------------------------------------------
# Display Evidence Analysis
# ------------------------------------------------------------

print("\nEvidence Utilization Analysis")
print("-" * 60)

display(evidence_usage_df)

# ------------------------------------------------------------
# Display Visualization Locations
# ------------------------------------------------------------

print("\nGenerated Visualizations")
print("-" * 60)

figure_files = [
    "outputs/evaluation/figures/evidence_usage_distribution.png",
    "outputs/evaluation/figures/text_length_comparison.png",
    "outputs/evaluation/figures/runtime_comparison.png",
]

for figure_file in figure_files:

    if os.path.exists(figure_file):
        file_size_kb = (
            os.path.getsize(figure_file)
            / 1024
        )

        print(
            f"{os.path.basename(figure_file):<40}"
            f"{file_size_kb:8.1f} KB"
        )

# ------------------------------------------------------------
# Executive Summary
# ------------------------------------------------------------

total_predictions = int(
    baseline_summary_df[
        baseline_summary_df["metric"]
        == "total_predictions"
    ]["value"].iloc[0]
)

unique_videos = int(
    baseline_summary_df[
        baseline_summary_df["metric"]
        == "unique_videos"
    ]["value"].iloc[0]
)

avg_evidence = float(
    baseline_summary_df[
        baseline_summary_df["metric"]
        == "average_evidence_records_per_sample"
    ]["value"].iloc[0]
)

avg_runtime = float(
    baseline_summary_df[
        baseline_summary_df["metric"]
        == "average_time_per_sample_seconds"
    ]["value"].iloc[0]
)

full_runtime = float(
    baseline_summary_df[
        baseline_summary_df["metric"]
        == "projected_full_dataset_runtime_hours"
    ]["value"].iloc[0]
)

print("\nExecutive Summary")
print("-" * 60)

print(
    f"Samples Evaluated                 : "
    f"{total_predictions:,}"
)

print(
    f"Unique Videos Evaluated           : "
    f"{unique_videos:,}"
)

print(
    f"Average Evidence Records/Sample   : "
    f"{avg_evidence:.2f}"
)

print(
    f"Average Runtime per Sample (sec)  : "
    f"{avg_runtime:.2f}"
)

print(
    f"Projected Full Dataset Runtime    : "
    f"{full_runtime:.2f} hours"
)

